# Speech Cheating Detection Pipeline
**Multi-signal ensemble: wav2vec2 (audio) + text features + pause features → XGBoost**

## Setup
1. Place audio folders (audios2/, audios4/) in the working directory
2. Place `gtlabels.csv` in the working directory
3. Download models from HuggingFace (cell below)
4. Run cells in order

In [ ]:
# Install dependencies (run once)
# !pip install onnxruntime librosa soundfile spacy xgboost pandas numpy scipy tqdm joblib faster-whisper
# !python -m spacy download en_core_web_sm

# For staging (GPU available):
# !pip install whisperx torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Config — edit these paths
AUDIO_ROOT = "."                           # folder containing audios2/, audios4/
LABELS_CSV = "gtlabels.csv"                # ground truth labels
ONNX_MODEL = "biased_wav2vec2_quant.onnx"  # wav2vec2 ONNX model
XGBOOST_MODEL = "xgboost_ensemble.json"    # XGBoost model
SCALER_PKL = "scaler.pkl"                  # feature scaler

# Auto-detect: GPU or CPU
import torch
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if HAS_GPU else ""))

## 1. Download Models from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download

repo_id = "Pransfrance/speechproj-models"

# remote_path -> local_path
files = {
    "biased/biased_wav2vec2_quant.onnx": ONNX_MODEL,
    "ensemble/xgboost_ensemble.json": XGBOOST_MODEL,
    "ensemble/scaler.pkl": SCALER_PKL,
    "ensemble/ensemble_results.json": "ensemble_results.json",
}

for remote, local in files.items():
    if os.path.exists(local):
        print(f"  Already exists: {local}")
        continue

    print(f"Downloading {remote}...")
    downloaded = hf_hub_download(repo_id=repo_id, filename=remote)
    # hf_hub_download returns the cached path — copy to local
    import shutil
    shutil.copy2(downloaded, local)
    print(f"  Saved: {local} ({os.path.getsize(local)/1e6:.1f} MB)")

# Verify all files exist
print("\nFile check:")
for remote, local in files.items():
    exists = os.path.exists(local)
    size = f"({os.path.getsize(local)/1e6:.1f} MB)" if exists else ""
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {local} {size}")

## 2. Build Manifest (scan audio files + join labels)

In [ ]:
from build_manifest import load_labels

# Load labels CSV — already has filepath, filename, folder, label
manifest = load_labels(LABELS_CSV)
manifest.to_csv("manifest.csv", index=False)

print(f"\nManifest: {len(manifest)} files")
print(f"  Cheating:     {(manifest['label_int']==1).sum()}")
print(f"  Not cheating: {(manifest['label_int']==0).sum()}")
print(f"\nBy batch:")
print(manifest.groupby("audio_batch")["label_int"].value_counts().unstack(fill_value=0))
manifest.head()

## 3. Transcribe Audio (WhisperX on GPU / faster-whisper on CPU)

In [ ]:
TRANSCRIPTS_FILE = "transcripts.json"

# Load existing transcripts if resuming
transcripts = {}
if os.path.exists(TRANSCRIPTS_FILE):
    with open(TRANSCRIPTS_FILE, encoding="utf-8") as f:
        transcripts = json.load(f)
    print(f"Resuming: {len(transcripts)} already transcribed")

# Filter to files that need transcription
to_transcribe = manifest[~manifest["filepath"].isin(transcripts.keys())]
print(f"Files to transcribe: {len(to_transcribe)}")

In [ ]:
if HAS_GPU:
    # ---- GPU path: WhisperX (fast, accurate word timestamps) ----
    import whisperx
    
    model = whisperx.load_model("large-v2", device=DEVICE, compute_type="float16", language="en")
    model_a, metadata = whisperx.load_align_model(language_code="en", device=DEVICE)
    
    for _, row in tqdm(to_transcribe.iterrows(), total=len(to_transcribe), desc="Transcribing (GPU)"):
        filepath = row["filepath"]
        try:
            audio = whisperx.load_audio(filepath)
            result = model.transcribe(audio, batch_size=8, language="en")
            if result.get("segments"):
                aligned = whisperx.align(result["segments"], model_a, metadata, audio,
                                        device=DEVICE, return_char_alignments=False)
                words = []
                text_parts = []
                for seg in aligned.get("segments", []):
                    text_parts.append(seg.get("text", "").strip())
                    for w in seg.get("words", []):
                        if "start" in w and "end" in w:
                            words.append({"word": w["word"], "start": round(w["start"], 3), "end": round(w["end"], 3)})
                transcripts[filepath] = {
                    "text": " ".join(text_parts), "words": words,
                    "filename": row["filename"], "duration_sec": round(len(audio)/16000, 2)
                }
            else:
                transcripts[filepath] = {"text": "", "words": [], "filename": row["filename"], "duration_sec": 0}
        except Exception as e:
            print(f"  ERROR {row['filename']}: {e}")
            transcripts[filepath] = {"text": "", "words": [], "filename": row["filename"], "duration_sec": 0}
        
        # Save every 10 files
        if len(transcripts) % 10 == 0:
            with open(TRANSCRIPTS_FILE, "w", encoding="utf-8") as f:
                json.dump(transcripts, f, indent=2, ensure_ascii=False)

else:
    # ---- CPU path: faster-whisper (slower but works without GPU) ----
    from faster_whisper import WhisperModel
    
    model = WhisperModel("medium", device="cpu", compute_type="int8")
    
    for _, row in tqdm(to_transcribe.iterrows(), total=len(to_transcribe), desc="Transcribing (CPU)"):
        filepath = row["filepath"]
        try:
            segments, info = model.transcribe(filepath, language="en", word_timestamps=True)
            words = []
            text_parts = []
            for seg in segments:
                text_parts.append(seg.text.strip())
                for w in (seg.words or []):
                    words.append({"word": w.word.strip(), "start": round(w.start, 3), "end": round(w.end, 3)})
            transcripts[filepath] = {
                "text": " ".join(text_parts), "words": words,
                "filename": row["filename"], "duration_sec": round(info.duration, 2)
            }
        except Exception as e:
            print(f"  ERROR {row['filename']}: {e}")
            transcripts[filepath] = {"text": "", "words": [], "filename": row["filename"], "duration_sec": 0}
        
        if len(transcripts) % 10 == 0:
            with open(TRANSCRIPTS_FILE, "w", encoding="utf-8") as f:
                json.dump(transcripts, f, indent=2, ensure_ascii=False)

# Final save
with open(TRANSCRIPTS_FILE, "w", encoding="utf-8") as f:
    json.dump(transcripts, f, indent=2, ensure_ascii=False)
print(f"\nDone! {len(transcripts)} transcripts saved to {TRANSCRIPTS_FILE}")

## 4. Extract Features

In [ ]:
FEATURES_CSV = "features_company.csv"

# Check if features already extracted
if os.path.exists(FEATURES_CSV):
    features_df = pd.read_csv(FEATURES_CSV)
    has_w2v = "wav2vec2_mean_p_read" in features_df.columns and features_df["wav2vec2_mean_p_read"].sum() > 0
    has_text = "filler_rate" in features_df.columns and features_df["filler_rate"].notna().sum() > 0
    has_prosodic = "f0_mean" in features_df.columns and features_df["f0_mean"].sum() > 0

    print(f"Found existing {FEATURES_CSV}: {len(features_df)} rows")
    print(f"  Text features:     {'YES' if has_text else 'NO'}")
    print(f"  Prosodic features: {'YES' if has_prosodic else 'NO'}")
    print(f"  wav2vec2 scores:   {'YES' if has_w2v else 'NO'}")

    if has_text and has_w2v:
        print("\nAll features present — skipping extraction. Delete features_company.csv to re-extract.")
    else:
        print("\nSome features missing — re-extracting...")
        os.remove(FEATURES_CSV)
        features_df = None
else:
    features_df = None

if features_df is None:
    from extract_features_company import (
        compute_text_features, compute_pause_features,
        compute_prosodic_features, compute_wav2vec2_scores,
        _empty_text_features, _empty_pause_features, _empty_prosodic_features
    )
    import onnxruntime as ort

    # Load wav2vec2 ONNX
    ort_session = ort.InferenceSession(ONNX_MODEL, providers=["CPUExecutionProvider"])
    print(f"Loaded ONNX model: {ONNX_MODEL}")

    # Build lookup from manifest for labels + batch info
    manifest_lookup = manifest.set_index("filename").to_dict("index")

    rows = []
    for filepath, t in tqdm(transcripts.items(), desc="Extracting features"):
        text = t.get("text", "")
        words = t.get("words", [])
        filename = t.get("filename", Path(filepath).name)

        text_feats = compute_text_features(text)
        pause_feats = compute_pause_features(words) if words else _empty_pause_features()
        prosodic_feats = compute_prosodic_features(filepath) if os.path.exists(filepath) else _empty_prosodic_features()
        wav2vec2_feats = compute_wav2vec2_scores(filepath, ort_session) if os.path.exists(filepath) else {"wav2vec2_read_ratio": 0, "wav2vec2_mean_p_read": 0, "wav2vec2_max_p_read": 0}

        # Get label + batch from manifest
        info = manifest_lookup.get(filename, {})
        label_int = info.get("label_int", -1)
        label_int = int(label_int) if pd.notna(label_int) else -1

        row = {"filepath": filepath, "filename": filename, "label_int": label_int,
               "audio_batch": info.get("audio_batch", "unknown"),
               "duration_sec": t.get("duration_sec", 0), "n_words": len(words)}
        row.update(text_feats)
        row.update(pause_feats)
        row.update(prosodic_feats)
        row.update(wav2vec2_feats)
        rows.append(row)

    features_df = pd.DataFrame(rows)
    features_df.to_csv(FEATURES_CSV, index=False)
    print(f"\nExtracted {len(features_df)} rows with {len(features_df.columns)} columns")

print(f"\nBy batch:")
print(features_df.groupby("audio_batch")["label_int"].value_counts().unstack(fill_value=0))
features_df.head()

## 5. Run Pretrained Ensemble (before finetuning)

In [ ]:
import xgboost as xgb
import joblib
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

# Load pretrained model
model = xgb.XGBClassifier()
model.load_model(XGBOOST_MODEL)
scaler = joblib.load(SCALER_PKL)

# Load results to get feature columns
results_path = None
for candidate in ["ensemble_results.json", "checkpoints_ensemble/ensemble_results.json",
                   "ensemble/ensemble_results.json"]:
    if os.path.exists(candidate):
        results_path = candidate
        break

if results_path:
    with open(results_path) as f:
        results_info = json.load(f)
    feature_cols = results_info["feature_columns"]
else:
    # Fallback: use all standard feature columns that exist in features_df
    print("WARNING: ensemble_results.json not found, using default feature list")
    from extract_features_company import compute_text_features, _empty_text_features
    ALL_DEFAULT = [
        "filler_rate", "filler_count", "repetition_rate", "repair_rate",
        "ttr", "mattr", "complex_word_rate", "avg_word_length",
        "n_words", "n_unique_words", "avg_sentence_length", "std_sentence_length",
        "fragment_rate", "n_sentences", "self_ref_rate", "discourse_marker_rate",
        "hedge_rate", "noun_rate", "verb_rate", "adj_rate",
        "pause_mean", "pause_std", "pause_median", "pause_skew",
        "long_pause_rate", "pause_ratio", "n_pauses", "pause_regularity",
        "pause_before_content_ratio", "pause_before_function_ratio",
        "mid_phrase_pause_rate", "words_per_sec", "articulation_rate",
        "f0_mean", "f0_std", "f0_range", "f0_skew", "f0_slope",
        "energy_mean", "energy_std", "speaking_rate_std",
        "wav2vec2_read_ratio", "wav2vec2_mean_p_read", "wav2vec2_max_p_read",
    ]
    feature_cols = [c for c in ALL_DEFAULT if c in features_df.columns]

# Filter to labeled data
labeled = features_df[features_df["label_int"].isin([0, 1])].copy()
print(f"Labeled samples: {len(labeled)} (cheating={(labeled['label_int']==1).sum()}, not cheating={(labeled['label_int']==0).sum()})")

# Predict with pretrained model
available_cols = [c for c in feature_cols if c in labeled.columns]
missing_cols = [c for c in feature_cols if c not in labeled.columns]
if missing_cols:
    print(f"Missing features (will be zero): {missing_cols}")
    for c in missing_cols:
        labeled[c] = 0

X = labeled[feature_cols].fillna(0).values
y = labeled["label_int"].values
X_scaled = scaler.transform(X)

y_pred = model.predict(X_scaled)
y_proba = model.predict_proba(X_scaled)[:, 1]

print(f"\n--- Pretrained Model on Company Data ---")
print(f"Accuracy: {accuracy_score(y, y_pred):.4f}")
print(f"F1:       {f1_score(y, y_pred):.4f}")
print(classification_report(y, y_pred, target_names=["not cheating", "cheating"]))
cm = confusion_matrix(y, y_pred)
print(f"Confusion Matrix:\n  TN={cm[0,0]}  FP={cm[0,1]}\n  FN={cm[1,0]}  TP={cm[1,1]}")

## 6. Finetune XGBoost on Company Data

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# Use all available features (including wav2vec2 scores)
ALL_FEATURE_COLS = feature_cols.copy()
for extra in ["wav2vec2_read_ratio", "wav2vec2_mean_p_read", "wav2vec2_max_p_read"]:
    if extra in features_df.columns and extra not in ALL_FEATURE_COLS:
        if features_df[extra].std() > 1e-8:
            ALL_FEATURE_COLS.append(extra)

print(f"Training with {len(ALL_FEATURE_COLS)} features")

# Split by folder: train on audios2, test on audios4
labeled = features_df[features_df["label_int"].isin([0, 1])].copy()

train_mask = labeled["audio_batch"] == "audios2"
test_mask = labeled["audio_batch"] == "audios4"

# If audio_batch column missing or no match, fall back to random split
if train_mask.sum() == 0 or test_mask.sum() == 0:
    print("WARNING: Could not split by audios2/audios4, falling back to random 80/20 split")
    from sklearn.model_selection import train_test_split
    train_idx, test_idx = train_test_split(labeled.index, test_size=0.2, random_state=42, stratify=labeled["label_int"])
    train_mask = labeled.index.isin(train_idx)
    test_mask = labeled.index.isin(test_idx)

train_df = labeled[train_mask]
test_df = labeled[test_mask]

print(f"\nTrain (audios2): {len(train_df)} (cheating={( train_df['label_int']==1).sum()}, not cheating={(train_df['label_int']==0).sum()})")
print(f"Test  (audios4): {len(test_df)} (cheating={(test_df['label_int']==1).sum()}, not cheating={(test_df['label_int']==0).sum()})")

X_train = train_df[ALL_FEATURE_COLS].fillna(0).values
y_train = train_df["label_int"].values
X_test = test_df[ALL_FEATURE_COLS].fillna(0).values
y_test = test_df["label_int"].values

# Scale
company_scaler = StandardScaler()
X_train_s = company_scaler.fit_transform(X_train)
X_test_s = company_scaler.transform(X_test)

# Train XGBoost
company_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=42,
)

company_model.fit(X_train_s, y_train,
                  eval_set=[(X_train_s, y_train), (X_test_s, y_test)],
                  verbose=20)

# Cross-validation on train set
cv_scores = cross_val_score(
    xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, eval_metric="logloss"),
    X_train_s, y_train, cv=5, scoring="f1"
)
print(f"\nCV F1 (train only): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

# Test results (audios4)
y_pred = company_model.predict(X_test_s)
y_proba = company_model.predict_proba(X_test_s)[:, 1]

print(f"\n--- Finetuned Model: trained on audios2, tested on audios4 ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1:       {f1_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=["not cheating", "cheating"]))
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n  TN={cm[0,0]}  FP={cm[0,1]}\n  FN={cm[1,0]}  TP={cm[1,1]}")

In [ ]:
# Feature importance
importances = company_model.feature_importances_
feat_imp = sorted(zip(ALL_FEATURE_COLS, importances), key=lambda x: -x[1])

print("Feature Importance (top 20):")
for name, imp in feat_imp[:20]:
    bar = "#" * int(imp * 100)
    print(f"  {name:<35s} {imp:.4f} {bar}")

# Threshold analysis
print(f"\nThreshold Sensitivity:")
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    from sklearn.metrics import precision_score, recall_score
    t_pred = (y_proba >= thresh).astype(int)
    t_prec = precision_score(y_test, t_pred, zero_division=0)
    t_rec = recall_score(y_test, t_pred, zero_division=0)
    t_f1 = f1_score(y_test, t_pred, zero_division=0)
    marker = " <-- default" if thresh == 0.5 else ""
    print(f"  threshold={thresh:.1f}: prec={t_prec:.4f} rec={t_rec:.4f} f1={t_f1:.4f}{marker}")

## 7. Save Finetuned Model

In [ ]:
# Save finetuned model
os.makedirs("checkpoints_finetuned", exist_ok=True)
company_model.save_model("checkpoints_finetuned/xgboost_company.json")
joblib.dump(company_scaler, "checkpoints_finetuned/scaler_company.pkl")

# Save config
config = {
    "feature_columns": ALL_FEATURE_COLS,
    "accuracy": round(accuracy_score(y_test, y_pred), 4),
    "f1": round(f1_score(y_test, y_pred), 4),
    "cv_f1_mean": round(cv_scores.mean(), 4),
    "n_train": len(y_train),
    "n_test": len(y_test),
    "feature_importances": {name: round(float(imp), 6) for name, imp in feat_imp},
}
with open("checkpoints_finetuned/results_company.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved to checkpoints_finetuned/")
print(f"  xgboost_company.json  — finetuned XGBoost")
print(f"  scaler_company.pkl    — feature scaler")
print(f"  results_company.json  — results + feature importance")

## 8. Predict on New (Unlabeled) Audio

In [ ]:
# Predict on all files (including unlabeled)
X_all_data = features_df[ALL_FEATURE_COLS].fillna(0).values
X_all_scaled = company_scaler.transform(X_all_data)

features_df["pred_cheating_prob"] = company_model.predict_proba(X_all_scaled)[:, 1]
features_df["pred_label"] = (features_df["pred_cheating_prob"] >= 0.5).astype(int)
features_df["pred_label_str"] = features_df["pred_label"].map({1: "cheating", 0: "not cheating"})

# Short audio fallback: if < 20 words, use wav2vec2 only
SHORT_AUDIO_MASK = features_df["n_words"] < 20
if SHORT_AUDIO_MASK.sum() > 0:
    print(f"Short audio fallback ({SHORT_AUDIO_MASK.sum()} files < 20 words): using wav2vec2 only")
    features_df.loc[SHORT_AUDIO_MASK, "pred_cheating_prob"] = features_df.loc[SHORT_AUDIO_MASK, "wav2vec2_mean_p_read"]
    features_df.loc[SHORT_AUDIO_MASK, "pred_label"] = (features_df.loc[SHORT_AUDIO_MASK, "wav2vec2_read_ratio"] >= 0.5).astype(int)
    features_df.loc[SHORT_AUDIO_MASK, "pred_label_str"] = features_df.loc[SHORT_AUDIO_MASK, "pred_label"].map({1: "cheating", 0: "not cheating"})

# Summary
print(f"\nPredictions:")
print(f"  Cheating:     {(features_df['pred_label']==1).sum()}")
print(f"  Not cheating: {(features_df['pred_label']==0).sum()}")

# Save predictions
output_cols = ["filename", "filepath", "duration_sec", "pred_label_str", "pred_cheating_prob", 
               "wav2vec2_read_ratio", "wav2vec2_mean_p_read", "filler_rate", "hedge_rate",
               "pause_before_content_ratio", "n_words"]
output_cols = [c for c in output_cols if c in features_df.columns]
features_df[output_cols].to_csv("predictions.csv", index=False)
print(f"\nSaved: predictions.csv")
features_df[output_cols].head(10)

## 9. Error Analysis

In [ ]:
# Analyze misclassifications on labeled data
labeled_with_pred = features_df[features_df["label_int"].isin([0, 1])].copy()
labeled_with_pred["correct"] = labeled_with_pred["label_int"] == labeled_with_pred["pred_label"]

errors = labeled_with_pred[~labeled_with_pred["correct"]]
print(f"Errors: {len(errors)} / {len(labeled_with_pred)} ({len(errors)/len(labeled_with_pred)*100:.1f}%)")

if len(errors) > 0:
    # False positives (flagged as cheating but actually not)
    fp = errors[errors["pred_label"] == 1]
    fn = errors[errors["pred_label"] == 0]
    print(f"\n  False positives (flagged innocent): {len(fp)}")
    print(f"  False negatives (missed cheater):   {len(fn)}")
    
    if len(fp) > 0:
        print(f"\n  FP avg features:")
        print(f"    wav2vec2_read_ratio: {fp['wav2vec2_read_ratio'].mean():.3f}")
        print(f"    filler_rate: {fp['filler_rate'].mean():.4f}")
        print(f"    hedge_rate: {fp['hedge_rate'].mean():.4f}")
    
    if len(fn) > 0:
        print(f"\n  FN avg features:")
        print(f"    wav2vec2_read_ratio: {fn['wav2vec2_read_ratio'].mean():.3f}")
        print(f"    filler_rate: {fn['filler_rate'].mean():.4f}")
        print(f"    hedge_rate: {fn['hedge_rate'].mean():.4f}")